# CV Masterclass 07: Metric Learning & Face Recognition

Welcome to Notebook 07. If you are building a facial recognition system for an airport, you cannot use a standard ImageNet ResNet classifier with a final Softmax layer.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"Why does a standard Softmax classifier completely fail in a Biometric Security system where a newly hired employee needs to be added to the database tomorrow?"*

---

## Table of Contents
1. **Closed-Set vs Open-Set:** The Softmax bottleneck.
2. **Metric Learning:** Learning distances instead of classes.
3. **Triplet Loss:** The Anchor, Positive, and Negative paradigm.
4. **Margin-based Softmax:** ArcFace and the hypersphere geometry.
5. **Production:** Using FAISS for sub-millisecond vector retrieval.


## 1. Closed-Set vs Open-Set

**Closed-Set Classification:** (e.g., ImageNet or MNIST). You have 1,000 fixed classes. You train the network to output a vector of length 1,000. You apply Softmax. You pick the highest probability. 
*   **The Problem:** If you are Apple building FaceID, you don't know the faces of the 100 Million users who will buy the iPhone next year. You cannot train a Softmax layer with 100 Million outputs.

**Open-Set Classification (Metric Learning):** Instead of learning a *classification*, you train the CNN to map an image of a face into a 512-dimensional vector. 
You mathematically enforce a geometry: If two images are of the same person, their 512-dimensional vectors must be extremely close together (Euclidean Distance or Cosine Similarity). If they are different people, the vectors must be far apart.

## 2. Triplet Loss

The original Google FaceNet (2015) used **Triplet Loss** to enforce this geometry.

You feed the network 3 images simultaneously:
1.  **Anchor ($A$):** A picture of Employee John.
2.  **Positive ($P$):** A completely different picture of Employee John.
3.  **Negative ($N$):** A picture of Employee Sarah.

**The Math:**
$L = \max(||f(A) - f(P)||^2 - ||f(A) - f(N)||^2 + \alpha, 0)$

The distance between John and John must be strictly smaller than the distance between John and Sarah, plus some safety margin $\alpha$.

## 3. Margin-based Softmax (ArcFace)

Triplet Loss works, but it causes a combinatorial explosion (selecting good $A, P, N$ triplets from a dataset of 1M images requires mining trillions of combinations).

Modern face recognition (like **ArcFace**) returns to the beloved Softmax equation, but hacks the underlying dot products.

Standard Softmax denominator uses $W^T x$. 
We know that $W^T x = ||W|| ||x|| \cos(\theta)$.

By mathematically forcing the Weights and the Features to have a magnitude of exactly $1.0$ (L2 Normalization), the dot product simply becomes $\cos(\theta)$—the angle between the feature and the class center on a perfect hypersphere.

**The ArcFace Magic:** ArcFace mathematically adds a strict angular penalty $m$ directly *inside* the cosine function during training for the correct class: $\cos(\theta + m)$.
This forces the network to mathematically squeeze the features for "John" incredibly tight together (to overcome the $+m$ penalty), creating massively separated, highly robust clusters on the hypersphere.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# -----------------------------------------------------
# IMPLEMENTATION: The ArcFace Margin
# -----------------------------------------------------

class ArcFace(nn.Module):
    def __init__(self, in_features, out_classes, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_classes = out_classes
        self.s = s  # Scaling factor
        self.m = m  # Angular margin penalty
        self.weight = nn.Parameter(torch.FloatTensor(out_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, labels):
        # 1. Normalize the features (x) and the weights (W)
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # 2. Extract the angle (theta) from the cosine
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        
        # 3. Add the margin penalty m ONLY to the correct ground-truth class
        target_logits = torch.cos(theta + self.m)
        
        # 4. Integrate the penalized target logits back into the original cosine matrix
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        
        # 5. Scale up to allow Softmax gradients to flow properly
        output *= self.s
        return output

print("ArcFace Angular Penalty Logic Defined.")
# This math powers almost all modern identity verification APIs.

## 4. Production Retrieval (FAISS)

Once you have an ArcFace model, you deploy it.
1. When a new employee is hired, you take 1 picture of them, run it through the CNN, extract the 512-dim vector, and save it in a database.
2. When they walk up to the door security camera, the camera extracts a 512-dim vector.

If your company has 100,000 employees, you must calculate the Cosine Distance between the live camera vector and all 100,000 database vectors instantly.

You cannot use an SQL loop to do this. You must use **FAISS (Facebook AI Similarity Search)**, which uses inverted file indexing (IVF) and Product Quantization (PQ) to search billions of vectors in milliseconds.